# Lesson 11 — Capstone: a tiny WAF with BOTH models

Lesson 11 — Capstone: a tiny WAF with BOTH models
==================================================

GURU NOTE
---------
Time to assemble everything. We'll build a miniature xorguard: two models scoring
the same traffic, thresholds calibrated for each, and the crucial 'defense in
depth' wiring where NEITHER model can block on its own. This is your repo in
miniature — every piece is a lesson you've already done:

  Lesson 1  features               Lesson 7  thresholds / calibration
  Lesson 4  logistic training      Lesson 9  MLP training (backprop)
  Lesson 6  class weights          Lesson 10 hashed n-gram featurization

The two models, mirroring the repo:
  * SUPERVISED (logistic) over hand-made indicator features  -> like supervised.json
  * NEURAL (MLP) over hashed char n-grams of the raw text     -> like neural.json

The decision policy, mirroring the repo:
  * DETECT (log it) if the fused score crosses the detect threshold.
  * PREVENT (403) ONLY if it also looks like a CRITICAL family (here: RCE).
    A high neural score on 'alert(' alone must NOT mint a 403. Same rule as the
    real engine.

Run me:  python lessons/lesson_11_capstone_mini_waf.py

## Setup

These notebooks *import* the real code from `lessons/*.py` — one source of truth, no forked copies. This cell finds that folder.

In [1]:
import sys, pathlib
for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (base / 'lessons' / '_util.py').exists():
        sys.path.insert(0, str(base / 'lessons'))
        print('lessons on path:', base / 'lessons'); break
else:
    raise RuntimeError('Open this notebook inside the xorguard-ml-course folder.')

lessons on path: /Users/anil/Documents/developer/vscodeworkspaces/xordev/xorguard-ml-course/lessons


## The key code (shown live from the source file)

Read it, then copy any function into a new cell and tweak it — the originals live in `lessons/lesson_11_capstone_mini_waf.py`.

In [2]:
import inspect
from lesson_11_capstone_mini_waf import indicator_features
for fn in (indicator_features,):
    print(inspect.getsource(fn)); print('-' * 72)

def indicator_features(text: str) -> np.ndarray:
    """Hand-made 'engine-style' indicators + bias. Stand-in for the 32-d vector."""
    t = text.lower()
    sqli = min(1.0, sum(t.count(k) for k in ["' or", "union", "select", "'='", "--"]) / 3)
    xss = min(1.0, sum(t.count(k) for k in ["<script", "onerror", "onload", "<svg", "<img"]) / 2)
    rce = min(1.0, sum(t.count(k) for k in ["cat ", "/etc/", "$(", "; ", "| "]) / 2)
    special = sum(c in "'\"<>;()=|&$/-" for c in text) / max(len(text), 1)
    return np.array([sqli, xss, rce, special, 1.0])

------------------------------------------------------------------------


## Run the full narrated lesson

This executes the same walkthrough as `python lessons/lesson_11_capstone_mini_waf.py`, printing each step with its result.

In [3]:
from lesson_11_capstone_mini_waf import main
main()


═════════════════════════════════════════════════
Lesson 11 — Capstone: a tiny WAF with BOTH models
═════════════════════════════════════════════════
    corpus: 660 requests  (360 attacks).
    split : 495 train / 165 validation (held out).

▸ Train model A — logistic regression on indicator features (Lessons 4/6).
    logistic AUC on validation = 1.000

▸ Train model B — MLP on hashed char n-grams (Lessons 9/10).
    featurizing text into 512-dim hashed vectors...
    training the net (watch accuracy climb)...
      epoch    1   acc=0.780
      epoch    5   acc=0.978
      epoch   25   acc=1.000
      epoch  100   acc=1.000
      epoch  120   acc=1.000
    neural AUC on validation = 1.000

▸ Fuse: the engine blends both scores. We'll average them (Lesson 7 to grade).
    fused AUC on validation = 1.000
    Two independent views (indicators + raw text) beat either alone,
    exactly why the engine keeps both models.

▸ Calibrate thresholds on validation (detect logs, prevent blocks).

## Exercises

Try these in new cells below. Worked answers are in `solutions/lesson_11_solution.py`.

```
1. Make XSS a critical family too (add 'xss' to CRITICAL) and re-grade. What
   happens to PREVENT coverage and to false blocks? This is a real policy lever.

2. Remove model B (score = p_log only) and re-grade the fused AUC and detect
   recall. Then remove model A instead. Which attacks does each model catch that
   the other misses? (Hint: the MLP sees raw text; the logistic sees indicators.)

3. Add a NEW attack family the models have never seen (e.g. a traversal payload
   "../../etc/passwd") to the demo list. Does the text-based MLP flag it even
   though no template trained on it? This is generalization from Lesson 10.
```

In [4]:
# your work here
